# 04 · ResNet feature-extraction smoke test

**OncoPlate v3.0.0 — implementation of the accepted v3.0 plan**

Run only after notebook 03. Start on a fitting-only subset; this is debugging, not a paper result.

This is research software, not a validated cancer-risk or chemical-detection product. Run cells in order. Missing independently collected data or approval records are genuine prerequisites, not permission to substitute synthetic results.

In [ ]:
from pathlib import Path
import os, sys, json
ON_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if ON_COLAB and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path(os.environ.get("ONCOPLATE_DRIVE_ROOT", "/content/drive/MyDrive/OncoPlate_Research"))
pointer = ROOT / ".oncoplate_install.json"
preferred = json.loads(pointer.read_text())["repository_path"] if pointer.exists() else str(ROOT / "oncoplate-research")
REPO = Path(os.environ.get("ONCOPLATE_REPO", preferred))
if not (REPO / "src/oncoplate").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src/oncoplate").exists(): REPO = candidate; break
assert (REPO / "src/oncoplate").exists(), "Run the supplied installer notebook or set ONCOPLATE_REPO to the extracted repository."
sys.path.insert(0, str(REPO / "src"))
from oncoplate.config import load_config, paths, initialize
from oncoplate.io import read_json, write_json, read_table, write_table, read_jsonl, write_jsonl, utcnow
cfg = load_config(REPO, root=ROOT, mode=os.environ.get("ONCOPLATE_MODE", "research"))
p = paths(cfg)
print("Dataset:", cfg["study"]["dataset"], "| Mode:", cfg["mode"], "| Persistent root:", cfg["root"])


## 1. Select at most 256 fitting images

In [ ]:
from oncoplate.pipeline import load_study
from oncoplate.governance import require_gate
require_gate(cfg,'supervisor_execution')
records,targets=load_study(cfg,'independent',stage_images=True)
smoke=records[records.split.eq('fit')].sort_values('record_id').head(256).copy()
assert not smoke.empty
print('Smoke images:',len(smoke))

## 2. Extract and persist reproducible shards

In [ ]:
from oncoplate.vision import backbone,feature_cache
import torch
model,dim=backbone('resnet50',pretrained=True)
SMOKE_BATCH_SIZE=32  # Reduce before the full run if the actual GPU needs it.
features,ids=feature_cache(smoke,model,p['features']/'resnet50_smoke',model_identity={'backbone':'resnet50','checkpoint':'IMAGENET1K_V2'},
 size=224,batch_size=SMOKE_BATCH_SIZE,workers=cfg['training']['num_workers'])
print(features.shape,dim)

## 3. Check ordering, finiteness and feature-cache resumption

In [ ]:
import numpy as np
assert features.shape==(len(smoke),dim)
assert np.isfinite(features).all()
assert list(ids)==smoke.record_id.tolist()
features2,ids2=feature_cache(smoke,model,p['features']/'resnet50_smoke',model_identity={'backbone':'resnet50','checkpoint':'IMAGENET1K_V2'},size=224,batch_size=SMOKE_BATCH_SIZE)
assert np.array_equal(features,features2)
report={'status':'passed','images':len(smoke),'feature_dim':dim,'cache_resume_equal':True,'training_performed':False}
write_json(p['reports']/"smoke_test.json",report);print(report)

## Completion and resumption
Outputs are written to the displayed persistent dataset-specific directories. Keep raw inputs, reviewer records, arrays and checkpoints private. Re-running a completed fit verifies its configuration rather than silently changing it. Use notebook 99 only after reviewing aggregate results; nothing here pushes to GitHub automatically.
